In [ ]:
from dotenv import load_dotenv
load_dotenv()  # Load environment variables BEFORE importing LangChain

# Initialize LangSmith tracing
import os
os.environ["LANGCHAIN_TRACING_V2"] = "true"

from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.agents import create_agent
from langchain.agents.middleware import wrap_tool_call
from langchain.messages import HumanMessage,ToolMessage
from langchain.tools.tool_node import ToolCallRequest
from langchain.tools import tool,ToolRuntime
from scripts import base_tools
from langsmith import Client

# Initialize LangSmith client
ls_client = Client()

In [ ]:
model = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0.7)

In [ ]:



@tool
def divide(a: int, b: int) -> int:
    """Divides two numbers."""
    return a / b

@tool
def get_message_count(runtime:ToolRuntime) -> int:
    """Returns the number of messages in the current conversation."""
    messages=runtime.state["messages"]
    context=runtime.context
    return f"for user {context.user_id} in session {context.session_id}, total messages so far: {len(messages)}"

In [ ]:
@wrap_tool_call
def monitor_tool_calls(request,handler):
    try:
        return handler(request)
        # Log the tool call to LangSmith
    except Exception as e:
        return ToolMessage(content=f"Error during tool call: {str(e)}",tool_call_id=request.tool_call_id)

In [ ]:
from dataclasses import dataclass


@dataclass
class UserContext:
      user_id: str
      session_id: str

In [ ]:
agent=create_agent(model=model, tools=[divide, get_message_count,base_tools.get_weather,base_tools.web_search],middleware=[monitor_tool_calls],context_schema=UserContext)
agent.invoke({'messages': [HumanMessage(content="total messages so far?")]},context=UserContext(user_id="user_123", session_id="session_456"))